# Tutorial 10 — Distributed Training: DDP & FSDP

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part III — Pretraining**  
**Follows:** Tutorial 9 (Pretraining: Mixed Precision, Schedules & Scaling Laws)  
**Precedes:** Tutorial 11 (Supervised Fine-Tuning & LoRA)

---

## What This Tutorial Covers

Distributed training is where most tutorials hand you a decorator and move on.
This one does not. It explains exactly what happens to gradients across
processes, what the collective communication operations are, why gradient
accumulation and DDP interact in a non-obvious way, and when FSDP is the
right choice over DDP.

Topics:

1. **The process group** — `torch.distributed`, ranks, world size, the init
   handshake. What actually happens when you call `dist.init_process_group`.
2. **All-reduce** — the collective operation that averages gradients across
   GPUs. [Ring-allreduce vs tree-allreduce.]{.underline} Why the communication is
   $O(\text{model size})$ not $O(\text{world size} \times \text{model size})$.
3. **`DistributedDataParallel` (DDP)** — what the wrapper does, the
   `no_sync()` context manager, the gradient accumulation interaction,
   the worker file-sharding fix from Tutorial 7 revisited.
4. **`FullyShardedDataParallel` (FSDP)** — when DDP runs out of memory,
   FSDP shards the model parameters themselves across GPUs. The forward/
   backward all-gather sequence. When to use which.
5. **`torchrun`** — the launcher. Environment variables. The single-node
   and multi-node invocation.
6. **A complete distributed pretraining script** that runs on 1–N GPUs
   with no code changes.

---

## 1. The Process Group

Distributed PyTorch runs your script as $W$ identical processes
simultaneously — one per GPU. Each process is assigned a **rank** (an integer
from 0 to $W-1$). Rank 0 is conventionally the "main" process that handles
logging, checkpointing, and printing.

Before any collective communication can happen, all processes must
synchronize via `init_process_group`. This establishes a shared
communication channel (the **backend**) and verifies that every expected
process has joined:

In [ ]:
import torch.distributed as dist

dist.init_process_group(
    backend='nccl',     # NCCL for GPU-to-GPU, 'gloo' for CPU or debugging
    # init_method and rank/world_size are read from environment variables
    # set by torchrun — you do not set them manually
)

rank       = dist.get_rank()         # this process's rank: 0, 1, 2, ...
world_size = dist.get_world_size()   # total number of processes
local_rank = int(os.environ['LOCAL_RANK'])  # rank on this machine
                                            # (differs from global rank in multi-node)

device = torch.device(f'cuda:{local_rank}')
torch.cuda.set_device(device)

**Why NCCL?** NCCL (NVIDIA Collective Communications Library) implements
GPU-to-GPU communication via NVLink (intra-node) or InfiniBand/Ethernet
(inter-node) without going through CPU memory. For GPU training it is
always faster than `gloo`, which routes communication through the CPU.
Use `gloo` only for debugging on CPU.

**The init handshake:** `init_process_group` is a **barrier** — it blocks
until all $W$ processes have called it. If one process crashes before calling
it, all others hang indefinitely. This is the most common cause of a
distributed job that appears to freeze immediately.

In [ ]:
# Canonical setup — call this at the top of every distributed script
import os
import torch
import torch.distributed as dist

def setup_distributed():
    """
    Initialize distributed training.
    Environment variables LOCAL_RANK, RANK, WORLD_SIZE are set by torchrun.
    Returns (rank, world_size, device, is_main).
    """
    dist.init_process_group(backend='nccl')

    rank       = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get('LOCAL_RANK', 0))
    device     = torch.device(f'cuda:{local_rank}')
    torch.cuda.set_device(device)
    is_main    = (rank == 0)

    if is_main:
        print(f"Distributed: world_size={world_size}  "
              f"backend={dist.get_backend()}")
    return rank, world_size, device, is_main


def cleanup_distributed():
    dist.destroy_process_group()

---

## 2. All-Reduce: The Collective Operation Behind DDP

In data-parallel training, every GPU holds a full copy of the model and
processes a different mini-batch. After the backward pass, each GPU has
a gradient tensor that reflects only its local batch. Before the optimizer
step, all GPUs must arrive at the *same* gradient — the average across
all local gradients:

$$\nabla W_{\text{global}} = \frac{1}{W} \sum_{i=0}^{W-1} \nabla W_i$$

The operation that computes this is [**all-reduce**]{.mark}: every process contributes
its tensor, the operation sums (or averages) them, and every process
receives the result. After all-reduce, every GPU has the same gradient.

### Ring-allreduce

The naïve implementation would have all processes send their gradient to
rank 0, rank 0 would sum them, and then broadcast the result back. This
is $O(W \times \text{model\_size})$ communication — it gets worse with
more GPUs.

Ring-allreduce is $O(2 \times \text{model\_size})$ regardless of $W$.
Processes are arranged in a ring. The operation runs in two phases:

**Phase 1 — Reduce-scatter** ($W-1$ rounds):  
Each process sends a chunk of its tensor to the next process in the ring
and receives a chunk from the previous process, accumulating the sum.
After $W-1$ rounds, each process holds the fully reduced value for one
chunk of the tensor.

**Phase 2 — All-gather** ($W-1$ rounds):  
Each process broadcasts its fully-reduced chunk around the ring. After
$W-1$ rounds, every process has the complete reduced tensor.

Total data transmitted per process: $2 \times \frac{W-1}{W} \times \text{model\_size} \approx 2 \times \text{model\_size}$.

```
Ring of 4 GPUs (ranks 0-1-2-3-0):

Before all-reduce:
  rank 0: [g0_A, g0_B, g0_C, g0_D]   (4 gradient chunks)
  rank 1: [g1_A, g1_B, g1_C, g1_D]
  rank 2: [g2_A, g2_B, g2_C, g2_D]
  rank 3: [g3_A, g3_B, g3_C, g3_D]

After reduce-scatter:
  rank 0: holds sum of chunk A from all ranks
  rank 1: holds sum of chunk B from all ranks
  rank 2: holds sum of chunk C from all ranks
  rank 3: holds sum of chunk D from all ranks

After all-gather:
  rank 0: [sum_A, sum_B, sum_C, sum_D]  ← complete averaged gradient
  rank 1: [sum_A, sum_B, sum_C, sum_D]
  rank 2: [sum_A, sum_B, sum_C, sum_D]
  rank 3: [sum_A, sum_B, sum_C, sum_D]
```

PyTorch's DDP wrapper handles all of this for you — but knowing it happens
explains why DDP's communication overhead scales with model size, not
with world size.

---

## 3. `DistributedDataParallel` (DDP)

DDP wraps your model. It registers a backward hook on every parameter that
fires when that parameter's gradient is ready. The hook launches an
all-reduce for that gradient. By the time the final backward pass is
complete, all gradients have been averaged across GPUs and every GPU is
ready to take the same optimizer step.

In [ ]:
from torch.nn.parallel import DistributedDataParallel as DDP

model = GPT(config).to(device)
model = DDP(model, device_ids=[local_rank])

# The underlying model is accessible via model.module
n_params = sum(p.numel() for p in model.module.parameters())

### The gradient accumulation interaction — the most common DDP bug

With gradient accumulation (Tutorial 9), you run $k$ forward/backward passes
before each optimizer step. DDP fires an all-reduce on every `.backward()` call[^ddp_hook].

[^ddp_hook]: DDP registers a gradient hook on each parameter. As soon as a parameter's gradient is ready during backprop, its all-reduce fires immediately (not waiting for the full backward pass). This overlaps communication with the backward computation of earlier layers, hiding most of the communication latency behind computation..
With accumulation steps of $k=4$, you are launching 4 all-reduces per optimizer
step when you only need 1. This wastes $3/4$ of your inter-GPU communication
bandwidth.

Worse: the all-reduce *averages* gradients across processes. If you accumulate
$k$ micro-steps and each triggers an all-reduce, the final gradient before
the optimizer step has been averaged $k$ times — not once. The gradient
scale is wrong.

The fix is `no_sync()` — a context manager that *disables gradient
synchronization* for all but the last accumulation step:

In [ ]:
# WRONG: all-reduce fires on every micro-step
for micro_step in range(accumulation_steps):
    x, y   = next(train_iter)
    _, loss = model(x, y)
    (loss / accumulation_steps).backward()   # all-reduce fires here — 4×
optimizer.step()

# CORRECT: all-reduce fires only on the final micro-step
for micro_step in range(accumulation_steps):
    # no_sync() suppresses the all-reduce for non-final steps
    is_last = (micro_step == accumulation_steps - 1)
    ctx     = model.no_sync() if not is_last else contextlib.nullcontext()
    with ctx:
        x, y   = next(train_iter)
        _, loss = model(x, y)
        (loss / accumulation_steps).backward()
optimizer.step()

`no_sync()` just sets a flag that defers the all-reduce. When the final
`backward()` runs outside `no_sync()`, the accumulated local gradients
from all $k$ steps are averaged across processes in one all-reduce.
Result: correct gradients, $1/k$ of the communication overhead.

### Data sharding across ranks

Every rank must see different data. If every rank sees the same batches,
training is equivalent to running on one GPU with $W$ times the compute
wasted. The `DocumentDataset` from Tutorial 7 shards by *file* across
DataLoader workers — we need to also shard by *rank*:

In [ ]:
class DistributedDocumentDataset(IterableDataset):
    """
    Like DocumentDataset, but also shards files across distributed ranks.
    Each rank reads a disjoint subset of files.
    Combined with worker sharding, every (rank, worker) pair reads
    a fully disjoint file subset.
    """

    def __init__(self, data_dir, split, rank, world_size):
        super().__init__()
        all_files  = sorted(Path(data_dir).glob(f'{split}_*.jsonl'))
        self.files = [f for i, f in enumerate(all_files)
                      if i % world_size == rank]
        if not self.files:
            raise RuntimeError(
                f"Rank {rank}: no files assigned. "
                f"Need at least {world_size} files for {split} split."
            )

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        files = self.files
        if worker_info is not None:
            files = [f for i, f in enumerate(files)
                     if i % worker_info.num_workers == worker_info.id]
        for path in files:
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        yield json.loads(line)['text']
                    except Exception:
                        continue

**Important:** with rank-level file sharding, every rank trains on a
different vocabulary of documents. The all-reduce averages their gradients
— this is exactly what we want. But if your corpus has too few files
(e.g., 1 file for 4 GPUs), some ranks get no data. Ensure your corpus
has at least $W \times \text{num\_workers}$ files, or use the shard-based
approach from Tutorial 7 which divides pre-tokenized shards among ranks.

---

## 4. `FullyShardedDataParallel` (FSDP)

DDP keeps a full copy of the model on every GPU. For a 10.7M parameter
nano model, this is trivial. For a 7B parameter model, a single copy
takes ~14GB in FP16 — times 8 GPUs, that is 112GB of GPU memory just
for model weights, before activations or optimizer state.

FSDP solves this by sharding the model parameters across GPUs. Each GPU
holds $1/W$ of the parameters at rest. When a layer is needed for
computation, FSDP runs an **all-gather** to reconstruct the full layer on
all GPUs, runs the computation, then discards the non-local shards. During
backward, it runs the same all-gather, computes gradients, and then a
**reduce-scatter** to average and re-shard the gradients.

```
DDP — memory per GPU:
  [full model] + [full optimizer state] + activations

FSDP — memory per GPU:
  [1/W of model] + [1/W of optimizer state] + activations

Memory saving: (W-1)/W of model + optimizer state
```

For 8 GPUs: FSDP uses 12.5% of the DDP per-GPU model memory. This is
what allows training 65B+ parameter models on clusters of commodity GPUs.

### The all-gather / reduce-scatter cycle

For each layer (called a **FSDP unit**) during the forward pass:

1. **All-gather**: every GPU receives the full parameter shard from all
   other GPUs → reconstructs the full layer parameters
2. **Compute**: forward pass through the layer
3. **Discard**: non-local parameter shards are freed from memory

During backward:

1. **All-gather**: reconstruct the full layer parameters again (needed
   to compute gradients with respect to inputs)
2. **Compute**: backward through the layer
3. **Reduce-scatter**: sum gradients across GPUs, scatter so each GPU
   holds the gradient for its shard only
4. **Discard**: non-local parameter/gradient shards freed

This means each layer's parameters traverse the network twice in the
backward pass (once for activation gradient, once for weight gradient),
but the memory footprint stays at $1/W$.

### Using FSDP

In [ ]:
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
)
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
import functools

# Define which modules are the FSDP units — typically one per Transformer block
# FSDP will all-gather / reduce-scatter at the boundary of each unit
auto_wrap = functools.partial(
    transformer_auto_wrap_policy,
    transformer_layer_cls={TransformerBlock},  # your block class from Tutorial 2
)

# Mixed precision policy — FP32 for params, BF16 for compute
mp_policy = MixedPrecision(
    param_dtype=torch.bfloat16,    # params stored as BF16 (halves memory)
    reduce_dtype=torch.bfloat16,   # gradient reduction in BF16
    buffer_dtype=torch.bfloat16,
)

model = GPT(config).to(device)
model = FSDP(
    model,
    auto_wrap_policy=auto_wrap,
    mixed_precision=mp_policy,
    sharding_strategy=ShardingStrategy.FULL_SHARD,  # shard params + grads + optimizer
    device_id=local_rank,
)

### [DDP vs FSDP: when to use which]{.underline}

| Condition | Use |
|---|---|
| Model fits on 1 GPU in FP16 | DDP — simpler, faster |
| Model requires 2–8 GPUs to fit | FSDP with `FULL_SHARD` |
| Fastest possible throughput, model fits | DDP |
| Saving optimizer memory only | FSDP with `SHARD_GRAD_OP` |
| Debugging distributed code | DDP — FSDP error messages are harder to read |
| Multi-node training | Both work; FSDP is more common at very large scale |

For our nano model (10.7M params, ~40MB FP32), DDP is always the right
choice. FSDP would add overhead for no benefit. We cover FSDP here so
you understand what you are opting into when you scale up.

---

## 5. `torchrun`: The Launcher

`torchrun` is PyTorch's distributed launcher. It starts $N$ copies of your
script, assigns each a rank, and sets the required environment variables:

```bash
# Single node, 2 GPUs
torchrun --nproc_per_node=2 train.py

# Single node, 4 GPUs
torchrun --nproc_per_node=4 train.py

# Two nodes, 4 GPUs each (8 GPUs total)
# Run on node 0:
torchrun \
    --nproc_per_node=4 \
    --nnodes=2 \
    --node_rank=0 \
    --master_addr=192.168.1.10 \
    --master_port=29500 \
    train.py

# Run on node 1:
torchrun \
    --nproc_per_node=4 \
    --nnodes=2 \
    --node_rank=1 \
    --master_addr=192.168.1.10 \
    --master_port=29500 \
    train.py
```

Environment variables set by `torchrun` (available in `os.environ`):

| Variable | Meaning |
|---|---|
| `RANK` | Global rank of this process (0 to world_size-1) |
| `LOCAL_RANK` | Rank on this machine (0 to nproc_per_node-1) |
| `WORLD_SIZE` | Total number of processes across all nodes |
| `MASTER_ADDR` | IP address of the rank-0 process |
| `MASTER_PORT` | Port used for the init rendezvous |

**Single-GPU fallback:** When running on one GPU (no `torchrun`), these
variables are not set. Your script must handle both cases:

In [ ]:
def is_distributed() -> bool:
    return 'RANK' in os.environ and int(os.environ.get('WORLD_SIZE', 1)) > 1

if is_distributed():
    rank, world_size, device, is_main = setup_distributed()
else:
    rank, world_size = 0, 1
    device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    is_main = True

---

## 6. Barriers and Only-on-Main Operations

Some operations must only happen once, on rank 0:
- Printing training progress
- Saving checkpoints
- Writing to the JSONL log
- Creating output directories

Other operations must happen on all ranks but need to be synchronized:
- Loading a checkpoint (all ranks must load the same weights)
- Evaluating (you want the average eval loss across all ranks' batches)

In [ ]:
# Only on main process
if is_main:
    print(f"step {step}: loss={loss:.4f}")
    logger.log_step(...)
    torch.save(checkpoint, path)

# Barrier: wait for rank 0 to finish writing checkpoint
# before other ranks try to read it
dist.barrier()

**The checkpoint load race condition:**

In [ ]:
# WRONG: all ranks try to save simultaneously → file corruption
torch.save(model.state_dict(), 'checkpoint.pt')

# CORRECT: only rank 0 saves; barrier before any rank loads
if is_main:
    torch.save(model.state_dict(), 'checkpoint.pt')
dist.barrier()   # all ranks wait here until rank 0 finishes
model.load_state_dict(torch.load('checkpoint.pt', map_location=device))

### Distributed evaluation

To get the correct eval loss, you want the average across all batches
processed by all ranks — not just rank 0's batches:

In [ ]:
@torch.no_grad()
def evaluate_distributed(model, val_loader, device, n_batches=20):
    model.eval()
    total_loss = torch.tensor(0.0, device=device)
    count      = torch.tensor(0,   device=device)

    for i, (x, y) in enumerate(val_loader):
        if i >= n_batches:
            break
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            _, loss = model(x.to(device), y.to(device))
        total_loss += loss
        count      += 1

    # All-reduce: sum loss and count across all ranks
    dist.all_reduce(total_loss, op=dist.ReduceOp.SUM)
    dist.all_reduce(count,      op=dist.ReduceOp.SUM)

    model.train()
    return (total_loss / count).item()

Without the all-reduce on eval, rank 0 reports only its own eval loss —
which fluctuates more and does not represent the full validation set.

---

## 7. DDP Checkpoint Saving

DDP wraps the model, adding a `.module` attribute. If you save
`model.state_dict()` directly, the keys will be prefixed with `module.`
— they won't load into a non-DDP model:

In [ ]:
# WRONG: saves keys like 'module.blocks.0.attn.W_q.weight'
torch.save(model.state_dict(), 'checkpoint.pt')

# CORRECT: unwrap DDP before saving
state_dict = model.module.state_dict() \
             if isinstance(model, DDP) else model.state_dict()
torch.save(state_dict, 'checkpoint.pt')

For FSDP, the full model is sharded across GPUs — you must use FSDP's own
checkpoint API to collect shards before saving:

In [ ]:
from torch.distributed.fsdp import FullStateDictConfig, StateDictType

# FSDP checkpoint: gather all shards to rank 0 and save
with FSDP.state_dict_type(
    model,
    StateDictType.FULL_STATE_DICT,
    FullStateDictConfig(offload_to_cpu=True, rank0_only=True),
):
    state_dict = model.state_dict()
    if is_main:
        torch.save(state_dict, 'checkpoint.pt')

---

## 8. The Complete Distributed Training Script

A single script that runs correctly on 1, 2, 4, or 8 GPUs with no
code changes:

In [ ]:
# distributed_pretrain.py
"""
Usage:
    # Single GPU (no torchrun)
    python distributed_pretrain.py

    # Multiple GPUs
    torchrun --nproc_per_node=4 distributed_pretrain.py
"""

import os
import contextlib
import math
import time
import json
import numpy as np
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

from tutorial_02 import GPT, NanoGPTConfig
from tutorial_07 import PretrainingDataset, make_dataloader
from tutorial_08 import make_cosine_schedule, TrainingConfig
from training_logger import TrainingLogger


# ------------------------------------------------------------------ #
#  Distributed helpers                                                 #
# ------------------------------------------------------------------ #

def is_distributed():
    return 'RANK' in os.environ and int(os.environ.get('WORLD_SIZE', 1)) > 1

def setup():
    if is_distributed():
        dist.init_process_group(backend='nccl')
        rank       = dist.get_rank()
        world_size = dist.get_world_size()
        local_rank = int(os.environ['LOCAL_RANK'])
        device     = torch.device(f'cuda:{local_rank}')
        torch.cuda.set_device(device)
    else:
        rank, world_size, local_rank = 0, 1, 0
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return rank, world_size, device, rank == 0

def cleanup():
    if is_distributed():
        dist.destroy_process_group()

def barrier():
    if is_distributed():
        dist.barrier()

def all_reduce_mean(tensor: torch.Tensor) -> torch.Tensor:
    if not is_distributed():
        return tensor
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    return tensor / dist.get_world_size()


# ------------------------------------------------------------------ #
#  Evaluation                                                          #
# ------------------------------------------------------------------ #

@torch.no_grad()
def evaluate(model, val_loader, device, n_batches=20):
    model.eval()
    dtype = torch.bfloat16 if device.type == 'cuda' else torch.float32

    total = torch.tensor(0.0, device=device)
    count = torch.tensor(0,   device=device)
    for i, (x, y) in enumerate(val_loader):
        if i >= n_batches:
            break
        with torch.autocast(device_type=device.type, dtype=dtype):
            _, loss = (model.module if isinstance(model, DDP) else model)(
                x.to(device), y.to(device)
            )
        total += loss.detach()
        count += 1

    if is_distributed():
        dist.all_reduce(total, op=dist.ReduceOp.SUM)
        dist.all_reduce(count, op=dist.ReduceOp.SUM)

    model.train()
    return (total / count).item()


# ------------------------------------------------------------------ #
#  Gradient stats                                                      #
# ------------------------------------------------------------------ #

def grad_stats(model):
    raw = model.module if isinstance(model, DDP) else model
    total_sq, ratios = 0.0, []
    for m in raw.modules():
        if isinstance(m, nn.Linear) and m.weight.grad is not None:
            g = m.weight.grad.norm().item()
            w = m.weight.norm().item()
            total_sq += g ** 2
            ratios.append(g / (w + 1e-8))
    gnorm = total_sq ** 0.5
    mean_r = float(np.mean(ratios)) if ratios else 0.0
    return gnorm, mean_r


# ------------------------------------------------------------------ #
#  Main training function                                              #
# ------------------------------------------------------------------ #

def train(cfg: TrainingConfig):
    rank, world_size, device, is_main = setup()
    dtype = torch.bfloat16 if device.type == 'cuda' else torch.float32

    if is_main:
        print(f"world_size={world_size}  device={device}  dtype={dtype}")
        Path(cfg.run_dir).mkdir(parents=True, exist_ok=True)

    # ---- Model ----
    model = GPT(cfg.model_config).to(device)
    if is_distributed():
        model = DDP(model, device_ids=[device.index])
    raw_model = model.module if isinstance(model, DDP) else model

    n_params = sum(p.numel() for p in raw_model.parameters())
    if is_main:
        print(f"Parameters: {n_params/1e6:.1f}M")

    # ---- Optimizer (with param group split for weight decay) ----
    decay   = [p for n, p in raw_model.named_parameters() if p.dim() >= 2]
    nodecay = [p for n, p in raw_model.named_parameters() if p.dim() <  2]
    optimizer = torch.optim.AdamW([
        {'params': decay,   'weight_decay': cfg.weight_decay},
        {'params': nodecay, 'weight_decay': 0.0},
    ], lr=cfg.max_lr, betas=(cfg.beta1, cfg.beta2))

    scheduler = make_cosine_schedule(
        optimizer, cfg.max_lr, cfg.min_lr,
        cfg.warmup_steps, cfg.max_steps
    )

    # ---- Data — each rank gets a different file shard ----
    from tutorial_03 import Tokenizer
    tok = Tokenizer.load('nano_tokenizer.json')

    # Build per-rank datasets
    train_ds = PretrainingDataset.from_jsonl(
        cfg.data_dir, tok, cfg.block_size,
        split='train', buffer_size=500,
        rank=rank, world_size=world_size,   # added in the distributed version
    )
    val_ds = PretrainingDataset.from_jsonl(
        cfg.data_dir, tok, cfg.block_size,
        split='val', buffer_size=100,
        rank=rank, world_size=world_size,
    )
    train_loader = make_dataloader(train_ds, cfg.batch_size, cfg.num_workers)
    val_loader   = make_dataloader(val_ds,   cfg.batch_size, 1)

    # ---- Logger (only rank 0 writes) ----
    logger = TrainingLogger(
        run_dir=cfg.run_dir,
        run_name=f'nano_gpt_ddp_w{world_size}',
        dashboard_url='http://localhost:8000' if is_main else None,
    ) if is_main else None

    # ---- Training loop ----
    model.train()
    train_iter   = iter(train_loader)
    total_tokens = 0

    for step in range(cfg.max_steps):
        t0 = time.time()
        optimizer.zero_grad()
        step_loss = 0.0

        for micro_step in range(cfg.accumulation):
            try:
                x, y = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                x, y = next(train_iter)

            x, y = x.to(device), y.to(device)
            total_tokens += x.numel()

            # Suppress all-reduce on non-final accumulation steps
            is_last = (micro_step == cfg.accumulation - 1)
            ctx     = model.no_sync() if (is_distributed() and not is_last) \
                      else contextlib.nullcontext()

            with ctx:
                with torch.autocast(device_type=device.type, dtype=dtype):
                    _, loss = model(x, y)
                (loss / cfg.accumulation).backward()
                step_loss += loss.item() / cfg.accumulation

        gnorm, mean_r = grad_stats(model)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step()

        if device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.time() - t0
        tps     = x.numel() * world_size / elapsed   # tokens/s across all GPUs
        lr      = optimizer.param_groups[0]['lr']

        # ---- Logging (rank 0 only) ----
        if is_main and step % cfg.log_every == 0:
            logger.log_step(
                step=step, train_loss=step_loss,
                learning_rate=lr, global_grad_norm=gnorm,
                tokens_per_sec=tps, mean_grad_ratio=mean_r,
                gpu_memory_gb=torch.cuda.memory_allocated()/1e9
                              if device.type == 'cuda' else 0.0,
            )

        # ---- Eval ----
        if step % cfg.eval_every == 0:
            eval_loss = evaluate(model, val_loader, device, cfg.eval_batches)

            if is_main:
                logger.log_step(
                    step=step, train_loss=step_loss,
                    learning_rate=lr, global_grad_norm=gnorm,
                    tokens_per_sec=tps, eval_loss=eval_loss,
                )
                print(
                    f"[rank0] step {step:5d}  "
                    f"train={step_loss:.4f}  eval={eval_loss:.4f}  "
                    f"lr={lr:.2e}  gnorm={gnorm:.3f}  "
                    f"tps={tps:,.0f}  world={world_size}"
                )

                # Save checkpoint (unwrap DDP)
                torch.save({
                    'step':      step,
                    'model':     raw_model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(),
                    'eval_loss': eval_loss,
                    'world_size': world_size,
                }, f'{cfg.run_dir}/checkpoint_step{step:05d}.pt')

            barrier()   # all ranks wait for rank 0 to finish writing

    # ---- Cleanup ----
    if is_main and logger:
        logger.close()
        print(f"\nTraining complete. Total tokens: {total_tokens * world_size:,}")

    cleanup()


if __name__ == '__main__':
    cfg = TrainingConfig(
        max_steps=5000,
        eval_every=500,
        accumulation=1,
        num_workers=2,
    )
    train(cfg)

---

## 9. Debugging Distributed Runs

### Hang on startup

Symptom: all processes start, then nothing happens.

Cause: [`init_process_group` is a barrier. If one process fails to reach it
(import error, data not found, OOM during model init), all others hang.]{.underline}

Fix: add prints *before* `init_process_group` to confirm all processes start:

```python
print(f"Process starting: rank={os.environ.get('RANK')}  pid={os.getpid()}")
dist.init_process_group(...)
print(f"Process group initialized: rank={rank}/{world_size}")
```

### Silent wrong gradients

[Symptom: training appears to work but loss does not improve as fast as
expected on multiple GPUs.]{.mark} Throughput scales but loss does not.

Cause: all ranks are seeing the same data (worker sharding bug) — gradients
are identical across ranks so the all-reduce has no effect.

Fix: print the first batch token sum from each rank at the start of training.
If they are all identical, the data sharding is broken:

```python
x, y = next(iter(train_loader))
print(f"rank {rank}: first batch token sum = {x.sum().item()}")
# All ranks should print different values
```

### OOM on rank 0 only

Symptom: rank 0 runs out of memory; other ranks do not.

Cause: rank 0 is doing extra work (logging, checkpoint saving, evaluation)
that allocates tensors that are never freed.

Fix: move evaluation tensors to CPU before accumulating, and use
`torch.no_grad()` + `del` explicitly for any large tensors created
in rank-0-only code paths.

### `nan` loss on some ranks but not others

Cause: one rank received a pathological batch (very rare tokens, very
long document that got split badly). The other ranks have healthy
gradients; after all-reduce, the nan propagates to all ranks.

Fix: use `safe_backward` from Tutorial 6 on every rank. Since `no_sync()`
defers the all-reduce, each rank can check its own gradients for NaN
before the collective operation:

In [ ]:
# After .backward() but before the final all-reduce step
for p in model.parameters():
    if p.grad is not None and torch.isnan(p.grad).any():
        print(f"rank {rank}: NaN gradient at step {step} — zeroing")
        p.grad.zero_()

Zeroing NaN gradients before all-reduce prevents the nan from
contaminating all ranks.

---

## Summary

| Concept | Key detail |
|---|---|
| `init_process_group` | Barrier — all ranks must reach it. Hangs if any rank dies first. |
| Ring-allreduce | $O(2 \times \text{model\_size})$ communication regardless of world size. |
| DDP backward hook | Fires all-reduce per parameter as gradients become ready during backward. |
| `no_sync()` | Suppresses all-reduce for non-final accumulation steps. Required for correct gradient scale. |
| Worker × rank sharding | Every (rank, worker) pair needs disjoint files. Verify with first-batch token sum. |
| DDP checkpoint | [Save `model.module.state_dict()` — not `model.state_dict()`.]{.underline} |
| FSDP vs DDP | DDP if model fits on 1 GPU. FSDP when you need to shard parameters across GPUs. |
| FSDP all-gather | Reconstructs full layer parameters before each forward/backward. Discards after. |
| `torchrun` env vars | `RANK`, `LOCAL_RANK`, `WORLD_SIZE`, `MASTER_ADDR`, `MASTER_PORT`. |
| Distributed eval | All-reduce loss and count across ranks before dividing — otherwise only rank 0's batches count. |
| Barrier before load | Rank 0 saves; `dist.barrier()`; all ranks load. Without barrier: race condition. |
| NaN in distributed | Zero NaN gradients per-rank before all-reduce — nan propagates through the collective. |

---

## Exercises

**1.** Run `distributed_pretrain.py` with `torchrun --nproc_per_node=1`
and `--nproc_per_node=2` (if you have 2 GPUs). Measure the tokens/sec
throughput in both cases. The 2-GPU run should be close to 2× the 1-GPU
throughput. If it is significantly less, the data pipeline is the
bottleneck — profile with `benchmark_dataloader` from Tutorial 7.

**2.** Reproduce the gradient accumulation / DDP bug deliberately:
modify the training loop to call `.backward()` *without* `no_sync()`
for all accumulation steps. Then compare gradient norms (before
optimizer step) against the correct `no_sync()` version over 100 steps.
Confirm they differ by a factor of approximately `accumulation_steps`.

**3.** Implement `verify_data_sharding(rank, loader, world_size)`:
collect the first 10 batch token-sums from each rank (you will need
a small test loop), then use `dist.all_gather` to collect all ranks'
results on rank 0, and assert that no two ranks produce the same sequence.
`dist.all_gather` signature:

```python
output = [torch.zeros_like(tensor) for _ in range(world_size)]
dist.all_gather(output, tensor)
```

**4.** Add mixed-precision `GradScaler` support to `distributed_pretrain.py`
for FP16. The scaler must be initialized on every rank. Verify that when
a NaN gradient occurs (inject one artificially), the scaler correctly
detects the inf/nan and skips the optimizer step on all ranks — not just
the rank that encountered the bad gradient.

**5.** Implement a `LinearScalingLR` wrapper: when training with $W$ GPUs,
the effective batch size is $W$ times larger (each optimizer step sees
$W \times \text{batch\_size}$ tokens). The linear scaling rule says the
LR should scale proportionally: `effective_lr = base_lr × world_size`.
Add `linear_scale_lr: bool = True` to `TrainingConfig` and apply the
scaling in `distributed_pretrain.py`. Verify that the loss curve with
`world_size=2` and linear-scaled LR matches the `world_size=1` baseline
more closely than without scaling.

**6.** Wrap the nano GPT with FSDP instead of DDP. Use
`transformer_auto_wrap_policy` with `TransformerBlock` as the unit class.
Save a checkpoint using `FULL_STATE_DICT` mode and verify it loads
correctly into a non-FSDP model. Report the per-GPU memory usage with
DDP vs FSDP for the nano model — with only 10.7M parameters, FSDP should
use *more* memory due to communication overhead, confirming that FSDP
only helps for large models.